[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/17_interpretabilidad_geometria/17_interpretabilidad_geometria.ipynb)

# Módulo 17 · Interpretabilidad y geometría de los embeddings

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración** | ≈ 5 h (lección ≈ 3 h + proyecto ≈ 2 h) |
| **GPU** | No necesaria con `FAST_DEV_RUN=True` (CPU, ≈ 3–4 min). T4/L4 solo si activas el embedder real opcional |
| **Prerrequisitos** | 09–10 (geometría, anisotropía, pooling), 11 (contrastivo), 12 (embedders), 15 (evaluación) |
| **Hilo conductor** | ¿Qué *sabe* el vector que Políglota guarda por cada documento? ¿Qué filtra? |

> 🔴 **Honestidad de madurez.** Este módulo mezcla técnicas **sólidas** (probing, direcciones lineales, anisotropía), **emergentes** (sparse autoencoders sobre embeddings) y **experimentales** (interpretación causal fiable, steering en producción). Marcamos cada una; lo que medimos con nuestros experimentos de juguete **no** es evidencia sobre modelos reales.
> 🧭 **Ruta núcleo vs ampliación.** Núcleo (≈ 3 h, lo que usa el proyecto): secciones 1–4 (probing, dirección lineal y steering), 6–7 (superposición y SAE TopK) y 9 (vec2text: el TODO 5 audita la inversión). Ampliación (≈ 2 h): 5 (dimensiones dominantes) y 8 (qué codifica un embedding: longitud y orden).

## 🎯 Objetivos de aprendizaje
1. **Diseñar e interpretar** sondas (*probes*) lineales y no lineales con **control tasks** y selectividad (Hewitt & Liang).
2. **Formular** la hipótesis de representación lineal y **comprobarla**: dirección de un concepto por diferencia de medias, steering de una consulta y borrado lineal de un concepto.
3. **Diagnosticar** dimensiones dominantes y anisotropía y **corregirlas** (estandarizar / *all-but-the-top*).
4. **Implementar desde cero** un *sparse autoencoder* **TopK** (con AuxK) en PyTorch y un SAE ReLU+L1, y **compararlos** recuperando características *plantadas* con verdad conocida.
5. **Aplicar** un SAE a embeddings de documentos de Políglota y **evaluar** si sus *features* son interpretables (pureza vs k-means/PCA) y estables entre semillas.
6. **Usar** una *feature* para **steering** del retrieval y **explicar** sus límites.
7. **Medir** qué información de un documento se puede **reconstruir** desde su embedding (inversión) y su coste de privacidad.

## 🔁 Conexión con módulos anteriores
<details><summary><b>1.</b> (Módulo 10) ¿Qué es la anisotropía de los embeddings contextuales y cómo se corrige?</summary>

Los vectores ocupan un cono estrecho (coseno medio alto entre textos no relacionados). Se corrige centrando, **whitening**, quitando componentes principales dominantes (*all-but-the-top*, Mu et al. 2018) o estandarizando dimensiones. En §5 medimos el efecto de una sola dimensión dominante.
</details>

<details><summary><b>2.</b> (Módulo 11) ¿Qué miden alineación y uniformidad?</summary>

Alineación: cercanía de pares positivos. Uniformidad: dispersión de los embeddings en la esfera. Un embedding anisotrópico tiene mala uniformidad: «menos capacidad» efectiva.
</details>

<details><summary><b>3.</b> (Módulo 12) ¿Qué es Matryoshka y qué sugiere sobre dónde está la información en un embedding?</summary>

El entrenamiento Matryoshka fuerza que **los primeros $m$ componentes** sirvan por sí solos: la información está *ordenada por importancia* en las dimensiones, lo cual no es el caso en un embedding «normal», donde está **distribuida** y rotada arbitrariamente.
</details>

<details><summary><b>4.</b> (Módulo 15) Antes de afirmar «el SAE A es mejor que B», ¿qué necesitas?</summary>

Misma métrica por query/feature, IC bootstrap, varias semillas y un test pareado; aquí veremos que **la semilla cambia qué features aparecen**.
</details>

<details><summary><b>5.</b> (Módulos 01 y 04, Diagnóstico) Un probe lineal predice el idioma (ES/EN/código) desde el embedding con un 99 % de acierto. ¿Demuestra que el modelo «entiende» el idioma?</summary>

No necesariamente: el idioma es casi trivial de leer en la **forma superficial** (acentos y «ñ» con bytes UTF-8 de 2 bytes, módulo 01; tokens distintos y fertilidad distinta, módulo 04; símbolos del código). Un probe alto solo dice que la información es linealmente accesible. Por eso la sección 3 mide la **selectividad** frente a una *control task* (Hewitt & Liang) antes de interpretar.
</details>

## 1. Intuición: abrir el vector
Un embedding de 768 números es una caja negra: sabemos que «cerca = similar», pero no **qué eje significa qué**. La interpretabilidad pregunta tres cosas, de menor a mayor exigencia:
1. **¿Qué información contiene?** (probing: ¿se puede *leer* X desde el vector?)
2. **¿Cómo está organizada?** (geometría: direcciones, dimensiones dominantes, subespacios)
3. **¿Qué usa el sistema?** (causalidad: si muevo el vector en esa dirección, ¿cambia el comportamiento?)

**Analogías con lo que ya sabes.** (a) En *recsys*, los factores latentes de una factorización de matrices rara vez son interpretables por sí solos: hay que *rotarlos* o inspeccionar sus ítems extremos; un SAE es el análogo moderno (diccionario disperso en lugar de bases densas). (b) En *compresión*, un diccionario sobrecompleto con codificación dispersa es *sparse coding*: más átomos que dimensiones, pero pocos activos por muestra.

> ### 🎨 GEMINI-IMAGEN · ID: `M17-IMG-3` · ⚠️ REEMPLAZAR ESTA CELDA
> **Capas de información dentro de un embedding de texto**
>
> **Qué muestra:** Mapa de qué codifica un embedding de documento (léxico, tema, idioma, estilo, longitud) y qué puede extraer un atacante o un probe.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Un documento pasa por un embedder y el vector resultante permite leer tema, idioma, palabras, longitud y orden, con una flecha de inversión hacia el texto original.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif legible, flechas claras, sin ilustraciones decorativas. 16:9.
> Composición: a la izquierda un documento (rectángulo blanco con borde gris y tres líneas de texto: «La caché expira tras un TTL…») con una flecha azul etiquetada «Embedder de texto» hacia un vector central dibujado como una fila larga de 12 celdas de colores (etiqueta «z ∈ R^768»).
> A la derecha del vector, cinco flechas verdes que salen hacia cinco cajas apiladas, con estos textos literales:
> «Tema (sonda lineal: muy legible)»,
> «Idioma (sonda lineal: casi perfecta)»,
> «Palabras concretas (parcial: inversión con vec2text)»,
> «Longitud y estilo (parcial)»,
> «Orden de las palabras (depende del modelo)».
> Debajo del vector, una flecha naranja hacia atrás hacia el documento con la etiqueta «Inversión: reconstruir el texto desde z» y un candado gris con el texto «privacidad (módulo 18)».
> Título superior: «¿Qué hay dentro de un embedding?». Pie en gris: «Probing mide lo que se puede leer, no lo que el sistema usa».
> Evita: texto ilegible o inventado, ilustraciones decorativas, sombras 3D, más de cinco cajas a la derecha, flechas ambiguas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M17-IMG-3.png` en la carpeta del módulo y reemplaza esta celda por: `![Un documento pasa por un embedder y el vector resultante permite leer tema, idioma, palabras, longitud y orden, con una flecha de inversión hacia el texto original.](assets/M17-IMG-3.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> documento -> [embedder] -> z (768-d) --+--> tema (muy legible)
>                                        +--> idioma (casi perfecto)
>                                        +--> palabras (parcial; inversión)
>                                        +--> longitud/estilo (parcial)
>                                        +--> orden de palabras (depende)
>                   z -- inversión --> texto   (privacidad, módulo 18)
> ```

> 👀 **Qué debes observar.** Un embedding **no es una caja opaca para un probe ni para un atacante**: tema e idioma se leen casi gratis, y las palabras concretas se pueden recuperar parcialmente. Lo que *no* sabemos todavía es qué parte de esa información **usa** el retrieval: eso exige experimentos causales (§4 y §8).

## 2. Entorno y embeddings de Políglota con etiquetas

In [ ]:
!pip install -q numpy scipy scikit-learn matplotlib pandas torch sentence-transformers

In [ ]:
import os, re, math, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42; FAST_DEV_RUN = True
random.seed(SEED); np.random.seed(SEED)
import torch
torch.manual_seed(SEED)
if FAST_DEV_RUN: torch.set_num_threads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
AZUL, NARANJA, VERDE, GRIS, ROJO = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280", "#D62728"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · semilla={SEED}")

Para **medir** interpretabilidad necesitamos una *verdad terreno*: usamos el generador sintético de Políglota, que etiqueta cada documento con **idioma, tema (12) y faceta (36)** (contrato común: mismo esquema que `carga_corpus`; aquí no usamos la cadena real porque necesitamos esas etiquetas; en el proyecto sí aplicarás el SAE a `carga_corpus`).
El «embedder» por defecto es **LSI-64** (TF-IDF + SVD): un embedding denso real, entrenado en el corpus y reproducible en CPU. Con red y GPU puedes sustituirlo por un embedder real (`USE_ST=True`): *esa opción no se pudo probar en el entorno de desarrollo*.

In [ ]:
# Corpus SINTÉTICO de Políglota (último recurso, mismo esquema que el real): 12 temas x 3 facetas.
# Por faceta: términos ES (5), EN (5), identificadores de código (3). Los 3 primeros términos son "centrales"
# (salen mucho en los documentos); los 2 últimos son sinónimos raros: las queries los usan -> hueco léxico.
TEMAS = {
 "cache": (["caché", "memoria", "datos"], ["cache", "memory", "data"], [
   ("ttl", "expiración|ttl|vencimiento|caducidad|refrescar", "expiration|ttl|timeout|lifetime|refresh", "set_ttl|expire_at|touch_key"),
   ("invalidacion", "invalidación|clave|purgar|obsoleto|desalojar", "invalidation|key|purge|stale|flush", "invalidate|purge_keys|delete_pattern"),
   ("evicion", "política|lru|desalojo|capacidad|recorte", "policy|lru|eviction|capacity|trim", "lru_evict|max_entries|pop_oldest")]),
 "basedatos": (["base", "datos", "consulta"], ["database", "data", "query"], [
   ("indices", "índice|b-tree|escaneo|selectividad|acelerar", "index|btree|scan|selectivity|speedup", "create_index|explain_plan|seq_scan"),
   ("transacciones", "transacción|commit|aislamiento|rollback|atomicidad", "transaction|commit|isolation|rollback|atomicity", "begin_tx|commit_tx|rollback_tx"),
   ("migraciones", "migración|esquema|columna|versionar|alterar", "migration|schema|column|versioning|alter", "run_migration|add_column|schema_version")]),
 "auth": (["usuario", "acceso", "seguridad"], ["user", "access", "security"], [
   ("tokens", "token|jwt|firma|sesión|testigo", "token|jwt|signature|session|bearer", "verify_jwt|sign_token|refresh_token"),
   ("oauth", "oauth|permiso|consentimiento|proveedor|delegar", "oauth|scope|consent|provider|delegate", "oauth_callback|request_scope|exchange_code"),
   ("hash", "contraseña|hash|sal|bcrypt|credencial", "password|hash|salt|bcrypt|credential", "hash_password|check_password|gen_salt")]),
 "contenedores": (["contenedor", "servicio", "despliegue"], ["container", "service", "deployment"], [
   ("imagenes", "imagen|capa|dockerfile|construir|empaquetar", "image|layer|dockerfile|build|package", "docker_build|pull_image|tag_image"),
   ("redes", "red|puerto|puente|exponer|conectar", "network|port|bridge|expose|connect", "create_network|publish_port|attach_bridge"),
   ("volumenes", "volumen|montaje|persistente|disco|compartir", "volume|mount|persistent|disk|share", "mount_volume|bind_path|prune_volumes")]),
 "testing": (["prueba", "código", "calidad"], ["test", "code", "quality"], [
   ("unitarias", "unitaria|aserción|fixture|función|verificar", "unit|assertion|fixture|function|verify", "assert_equal|run_unit|make_fixture"),
   ("mocks", "simulacro|mock|dependencia|sustituir|doble", "mock|stub|dependency|replace|double", "patch_call|make_mock|reset_mock"),
   ("cobertura", "cobertura|línea|rama|informe|medir", "coverage|line|branch|report|measure", "run_coverage|branch_report|min_cover")]),
 "concurrencia": (["hilo", "tarea", "ejecución"], ["thread", "task", "execution"], [
   ("hilos", "hilo|gil|paralelo|pool|lanzar", "thread|gil|parallel|pool|spawn", "start_thread|join_all|pool_map"),
   ("asincrono", "asíncrono|corrutina|await|bucle|esperar", "async|coroutine|await|loop|wait", "run_loop|gather_tasks|await_all"),
   ("bloqueos", "bloqueo|candado|interbloqueo|exclusión|sincronizar", "lock|mutex|deadlock|exclusion|synchronize", "acquire_lock|release_lock|try_lock")]),
 "api": (["api", "petición", "recurso"], ["api", "request", "resource"], [
   ("paginacion", "paginación|cursor|página|límite|desplazamiento", "pagination|cursor|page|limit|offset", "next_cursor|page_size|fetch_page"),
   ("versionado", "versión|compatibilidad|ruta|migrar|deprecar", "version|compatibility|route|migrate|deprecate", "api_version|route_v2|sunset_header"),
   ("limitacion", "cuota|tasa|límite|429|estrangular", "quota|rate|limit|429|throttle", "rate_limit|retry_after|token_bucket")]),
 "observabilidad": (["sistema", "registro", "evento"], ["system", "log", "event"], [
   ("logs", "log|nivel|formato|rotar|mensaje", "log|level|format|rotate|message", "get_logger|set_level|rotate_logs"),
   ("metricas", "métrica|contador|histograma|panel|alerta", "metric|counter|histogram|dashboard|alert", "inc_counter|observe_hist|export_metrics"),
   ("trazas", "traza|span|latencia|distribuido|correlación", "trace|span|latency|distributed|correlation", "start_span|inject_trace|trace_id")]),
 "git": (["repositorio", "cambio", "rama"], ["repository", "change", "branch"], [
   ("ramas", "rama|checkout|puntero|crear|cambiar", "branch|checkout|pointer|create|switch", "git_branch|git_checkout|list_branches"),
   ("merge", "fusión|merge|conflicto|resolver|integrar", "merge|conflict|resolve|integrate|combine", "git_merge|resolve_conflict|abort_merge"),
   ("rebase", "rebase|historial|reescribir|reaplicar|linealizar", "rebase|history|rewrite|replay|linearize", "git_rebase|squash_commits|pick_commit")]),
 "ml": (["modelo", "entrenamiento", "datos"], ["model", "training", "data"], [
   ("gradiente", "gradiente|descenso|pérdida|paso|optimizar", "gradient|descent|loss|step|optimize", "grad_step|zero_grad|clip_norm"),
   ("sobreajuste", "sobreajuste|regularización|generalizar|memorizar|dropout", "overfitting|regularization|generalize|memorize|dropout", "add_dropout|l2_penalty|early_stop"),
   ("validacion", "validación|pliegue|cruzada|partición|estimar", "validation|fold|cross|split|estimate", "kfold_split|val_score|holdout")]),
 "red": (["red", "paquete", "conexión"], ["network", "packet", "connection"], [
   ("tcp", "tcp|segmento|retransmisión|ventana|fiable", "tcp|segment|retransmission|window|reliable", "tcp_connect|set_window|resend_segment"),
   ("dns", "dns|dominio|resolución|registro|nombre", "dns|domain|resolution|record|name", "resolve_host|lookup_a|flush_dns"),
   ("tls", "tls|certificado|cifrado|apretón|confianza", "tls|certificate|encryption|handshake|trust", "tls_handshake|load_cert|verify_chain")]),
 "memoria": (["memoria", "proceso", "objeto"], ["memory", "process", "object"], [
   ("gc", "recolector|basura|referencia|ciclo|liberar", "collector|garbage|reference|cycle|free", "gc_collect|weak_ref|ref_count"),
   ("virtual", "virtual|página|dirección|paginación|fallo", "virtual|page|address|paging|fault", "map_page|page_fault|swap_out"),
   ("fugas", "fuga|crecimiento|perfilar|heap|retención", "leak|growth|profile|heap|retention", "heap_snapshot|track_alloc|find_leak")]),
}

In [ ]:
import os, random, re

PLANT_ES = ["Esta guía explica cómo {a} influye en {b} cuando se trabaja con {c}.",
            "Para resolver el problema de {a}, revisa primero {b} y luego {c}.",
            "En producción conviene vigilar {a}, porque {b} cambia el comportamiento de {c}.",
            "Un error habitual es mezclar {a} con {b} sin entender {c}.",
            "La documentación recomienda medir {a} antes de tocar {b} o {c}.",
            "Ejemplo práctico: ajustar {a} reduce el coste de {b} en {c}."]
PLANT_EN = ["This guide explains how {a} affects {b} when working with {c}.",
            "To solve the problem with {a}, first check {b} and then {c}.",
            "In production you should watch {a}, because {b} changes the behavior of {c}.",
            "A common mistake is mixing {a} with {b} without understanding {c}.",
            "The documentation recommends measuring {a} before touching {b} or {c}.",
            "Practical example: tuning {a} lowers the cost of {b} in {c}."]
QS_ES = ["cómo gestionar {s}", "problemas con {s}", "guía de {s}", "qué hacer con {s}"]
QS_EN = ["how to handle {s}", "problems with {s}", "guide to {s}", "what to do about {s}"]

def _slot(rng, core, rare, shared, other):
    r = rng.random()
    return rng.choice(core) if r < .62 else rng.choice(shared) if r < .80 else rng.choice(rare) if r < .86 else rng.choice(other)

def genera_sintetico(n_temas: int = 8, docs_por_faceta: int = 4, seed: int = 42) -> dict:
    """Mismo esquema que el corpus real. Relevancia: docs de la misma faceta e idioma (code: faceta, queries 'code')."""
    rng = random.Random(seed)
    nombres = list(TEMAS)[:n_temas]
    docs, queries, qrels = [], [], {}
    for t in nombres:
        es_sh, en_sh, facetas = TEMAS[t]
        for fid, es, en, code in facetas:
            otros = [x for n in nombres if n != t for x in TEMAS[n][2][0][1].split("|")[:3]]
            otros_en = [x for n in nombres if n != t for x in TEMAS[n][2][0][2].split("|")[:3]]
            es, en, code = es.split("|"), en.split("|"), code.split("|")
            rel = {"es": {}, "en": {}, "code": {}}
            for lang, terms, shared, oth, plant in (("es", es, es_sh, otros, PLANT_ES), ("en", en, en_sh, otros_en, PLANT_EN)):
                for i in range(docs_por_faceta):
                    sents = [rng.choice(plant).format(a=_slot(rng, terms[:3], terms[3:], shared, oth),
                                                      b=_slot(rng, terms[:3], terms[3:], shared, oth),
                                                      c=_slot(rng, terms[:3], terms[3:], shared, oth)) for _ in range(3)]
                    did = f"{lang}-{t}.{fid}-{i}"
                    docs.append({"id": did, "lang": lang, "text": " ".join(sents)}); rel[lang][did] = 1
            for i in range(max(2, docs_por_faceta - 1)):
                fn = rng.choice(code); aux = rng.choice(code); w = rng.choice(en[:3])
                did = f"code-{t}.{fid}-{i}"
                docs.append({"id": did, "lang": "code", "text": f"def {fn}(x, n={rng.randint(2, 9)}):\n    \"\"\"{w} helper for {rng.choice(en_sh)}\"\"\"\n    return {aux}(x, {rng.choice(code)})"})
                rel["code"][did] = 1
            for lang, terms, qs in (("es", es, QS_ES), ("en", en, QS_EN), ("code", en, QS_EN)):
                s = " ".join(([rng.choice(terms[:3])] if rng.random() < .5 else []) + rng.sample(terms[3:], 2))   # 50%: 1 central + 2 raros
                qid = f"q-{lang}-{t}.{fid}"
                queries.append({"qid": qid, "lang": lang, "text": rng.choice(qs).format(s=s) + (" function" if lang == "code" else "")})
                qrels[qid] = dict(rel[lang])
    rng.shuffle(docs)
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "synthetic"}

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

def tok_simple(text: str) -> list[str]:
    """Token léxico: minúsculas + regex de palabras (snake_case queda como UN solo token; lo veremos fallar en código)."""
    return re.findall(r"\w+", text.lower())

def tok_code(text: str) -> list[str]:
    """Igual, pero parte snake_case y camelCase (set_ttl -> set, ttl). Es tokenización con criterio de dominio."""
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text).replace("_", " ")
    return re.findall(r"[^\W_]+", text.lower())

class BM25:
    """BM25 (Robertson/Lucene idf) con matriz dispersa. k1=1.2, b=0.75."""
    def __init__(self, tokenizer=tok_simple, k1=1.2, b=0.75):
        self.tok, self.k1, self.b = tokenizer, k1, b
    def fit(self, texts, ids):
        self.ids = list(ids)
        self.cv = CountVectorizer(tokenizer=self.tok, lowercase=False, token_pattern=None)
        X = self.cv.fit_transform(texts).tocsr().astype(float)
        self.N = X.shape[0]; self.dl = np.asarray(X.sum(1)).ravel(); self.avg = self.dl.mean()
        df = np.asarray((X > 0).sum(0)).ravel()
        self.idf = np.log(1 + (self.N - df + 0.5) / (df + 0.5))
        self.X = X.tocsc()
        return self
    def scores(self, query: str) -> np.ndarray:
        s = np.zeros(self.N)
        for j in set(self.cv.transform([query]).indices):
            col = self.X[:, j].toarray().ravel()
            s += self.idf[j] * col * (self.k1 + 1) / (col + self.k1 * (1 - self.b + self.b * self.dl / self.avg))
        return s

class LSI:
    """Retriever 'denso' sin descargas: TF-IDF sublineal + SVD truncada (LSA) + coseno. Hace de embedder de CPU."""
    def __init__(self, dim=48, seed=42):
        self.dim, self.seed = dim, seed
    def fit(self, texts, ids):
        self.ids = list(ids)
        self.tf = TfidfVectorizer(sublinear_tf=True)
        T = self.tf.fit_transform(texts)
        self.svd = TruncatedSVD(min(self.dim, T.shape[1] - 1), random_state=self.seed)
        self.Z = self._norm(self.svd.fit_transform(T))
        return self
    @staticmethod
    def _norm(Z): return Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)
    def scores(self, query: str) -> np.ndarray:
        return self.Z @ self._norm(self.svd.transform(self.tf.transform([query])))[0]

def rrf_scores(score_lists, k: int = 60) -> np.ndarray:
    """Reciprocal Rank Fusion: sum 1/(k + rango). Ignora la escala de cada score (ventaja y límite)."""
    out = np.zeros(len(score_lists[0]))
    for s in score_lists:
        order = np.lexsort((np.arange(len(s)), -np.asarray(s)))
        out[order] += 1.0 / (k + np.arange(len(s)) + 1)
    return out

def build_run(scorer, queries, doc_ids, topn: int = 100) -> dict:
    """scorer(texto_query) -> vector de scores alineado con doc_ids. Devuelve run {qid: [doc_ids top-n]}."""
    return {q["qid"]: rank_from_scores(scorer(q["text"]), doc_ids, topn) for q in queries}

In [ ]:
corp = genera_sintetico(n_temas=12, docs_por_faceta=40 if FAST_DEV_RUN else 80, seed=SEED)
docs = [d for d in corp["docs"] if d["lang"] != "code"]; txt = [d["text"] for d in docs]; ids = [d["id"] for d in docs]
lang = np.array([d["lang"] for d in docs]); facet = np.array([d["id"].split("-", 1)[1].rsplit("-", 1)[0] for d in docs]); topic = np.array([f.split(".")[0] for f in facet])
uniq = lambda a: np.unique(a, return_inverse=True)[1]; lang_id, top_id, fac_id = uniq(lang), uniq(topic), uniq(facet)

USE_ST = False       # True: embedder real (sentence-transformers multilingüe). NO probado en el entorno de desarrollo (sin HF).
def embebe_docs(textos, dim=64):
    if USE_ST:
        try:
            from sentence_transformers import SentenceTransformer
            return SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device).encode(textos, normalize_embeddings=True, show_progress_bar=False).astype(np.float32), None
        except Exception as e: print("ST no disponible:", type(e).__name__)
    m = LSI(dim, SEED).fit(textos, ids); return m.Z.astype(np.float32), m

E, lsi = embebe_docs(txt, 64); En = E / np.linalg.norm(E, axis=1, keepdims=True)
print(f"{len(docs)} documentos · {len(set(facet))} facetas · {len(set(topic))} temas · embeddings {E.shape}")

## 3. Probing: ¿se puede *leer* X desde el vector?
Una **sonda** es un clasificador supervisado entrenado sobre embeddings **congelados** para predecir una propiedad $y$ (idioma, tema, categoría gramatical…). Si la sonda acierta, la información está *presente* (y accesible para esa familia de sondas).
**Hewitt & Manning (NAACL 2019)** llevaron la idea más lejos con la **sonda estructural**: buscan una transformación lineal $B$ tal que la **distancia L2 al cuadrado** $\lVert B(h_i-h_j)\rVert^2$ aproxime la distancia en el árbol sintáctico entre las palabras $i,j$ (y $\lVert Bh_i\rVert^2$ la profundidad); en ELMo y BERT existen esas transformaciones y en los baselines no: **el árbol sintáctico está incrustado en la geometría**.

**El problema:** una sonda muy expresiva puede *aprender la tarea por sí sola* aunque el embedding no la codifique. **Hewitt & Liang (EMNLP 2019)** proponen **control tasks**: asignar etiquetas **aleatorias por tipo** (aquí, por faceta). Una sonda que memoriza tipos saca buena nota en la tarea de control; una sonda *selectiva* saca nota alta en la tarea real y baja en el control:
$$\text{selectividad}=\text{acc}_{\text{tarea real}}-\text{acc}_{\text{control}}$$
Belinkov (2022) resume las limitaciones: probar que *se puede leer* no prueba que el modelo *lo use*.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

def sonda(X, y, tipo="lineal", seed=42):
    """Precisión en test de una sonda (clasificador) entrenada sobre embeddings congelados."""
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=None)
    m = LogisticRegression(max_iter=2000) if tipo == "lineal" else MLPClassifier((256,), max_iter=400, random_state=seed)
    return float(m.fit(Xa, ya).score(Xb, yb))

def tarea_control(grupos, n_clases, seed=42):
    """Control task (Hewitt & Liang 2019): etiqueta ALEATORIA fija por cada 'tipo' (aquí: por faceta). Una sonda que memoriza tipos
    saca buena nota sin que el embedding codifique nada lingüístico -> selectividad = acc(tarea real) - acc(control)."""
    rng = np.random.default_rng(seed); tabla = {g: int(rng.integers(0, n_clases)) for g in sorted(set(grupos))}
    return np.array([tabla[g] for g in grupos])

def diagnostico_dimensiones(E):
    """Estadísticos de 'dimensiones dominantes' (Timkey & van Schijndel 2021): fracción de varianza de la dimensión top,
    coseno medio entre pares al azar (anisotropía) y el mismo coseno tras estandarizar cada dimensión."""
    E = np.asarray(E, dtype=np.float64); var = E.var(0); rng = np.random.default_rng(0); i, j = rng.integers(0, len(E), (2, 4000))
    cos = lambda A: float(np.mean(np.sum(A[i] * A[j], 1) / (np.linalg.norm(A[i], axis=1) * np.linalg.norm(A[j], axis=1) + 1e-12)))
    Z = (E - E.mean(0)) / (E.std(0) + 1e-12)
    return {"var_top1": float(var.max() / var.sum()), "var_top3": float(np.sort(var)[-3:].sum() / var.sum()), "coseno_medio": cos(E), "coseno_medio_estandarizado": cos(Z)}

In [ ]:
E24, _ = embebe_docs(txt, 24)                         # embedding compacto: si es muy grande, hasta el control es separable
ctrl = tarea_control(list(facet), 12)                 # etiqueta aleatoria (12 clases) fija por faceta
filas = []
for tarea, y in (("idioma (2)", lang_id), ("tema (12)", top_id), ("CONTROL (12, aleatorio por faceta)", ctrl)):
    filas.append({"tarea": tarea, "sonda lineal": sonda(E24, y, "lineal"), "sonda MLP-256": sonda(E24, y, "mlp")})
pt = pd.DataFrame(filas).set_index("tarea")
sel = pd.Series({"selectividad lineal": pt.loc["tema (12)", "sonda lineal"] - pt.loc["CONTROL (12, aleatorio por faceta)", "sonda lineal"],
                 "selectividad MLP": pt.loc["tema (12)", "sonda MLP-256"] - pt.loc["CONTROL (12, aleatorio por faceta)", "sonda MLP-256"]})
display(pt.round(3))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6)); pt.plot(kind="bar", ax=ax[0], color=[AZUL, NARANJA], rot=12); ax[0].set_ylabel("precisión en test"); ax[0].set_title("Sondas sobre LSI-24"); ax[0].legend(frameon=False, fontsize=8)
ax[1].bar(sel.index, sel.values, color=[VERDE, ROJO]); ax[1].set_title("Selectividad (tema − control)"); ax[1].set_ylabel("selectividad"); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Idioma y tema se leen casi perfectamente. Pero la sonda **MLP** también «lee» la tarea de control (≈ 0.95): memoriza, así que su selectividad es mínima; la sonda **lineal** **no** puede (≈ 0.55) y su selectividad es alta. Conclusión metodológica (Hewitt & Liang): **prefiere sondas simples y reporta siempre la tarea de control**; si no, el «ha aprendido sintaxis» puede ser «la sonda aprendió sintaxis».
> Si subes la dimensión a 64+, hasta el control se vuelve linealmente separable (el embedding es tan grande que *cualquier* etiquetado por clúster es lineal): la selectividad también depende de la **capacidad** del espacio.

## 4. La hipótesis de representación lineal: direcciones, steering y borrado
**Hipótesis (Park, Choe, Veitch; ICML 2024):** los conceptos se codifican como **direcciones** del espacio de representación. Formalizan «concepto lineal» mediante pares contrafactuales y relacionan la hipótesis con (i) *probing lineal* y (ii) *steering* (mover la representación en esa dirección); argumentan que la geometría correcta para medir similitud entre direcciones requiere un **producto interno causal**, no el coseno ingenuo.
**Receta mínima:** dirección de un concepto por **diferencia de medias**: $v_{A\to B}=\bar z_B-\bar z_A$. **Steering:** $q'=\mathrm{norm}(q+\alpha v_{A\to B})$.

✍️ **Ejemplo a mano (2D).** Documentos de «cache»: $(1, 0.2)$ y $(0.8, 0.6)$, media $(0.9, 0.4)$; de «git»: $(0.2, 1)$ y $(0.6, 0.8)$, media $(0.4, 0.9)$. Dirección $v_{\text{cache}\to\text{git}} = (-0.5, 0.5)$. La query $q = (1, 0.2)$ con $\alpha = 1$ pasa a $(0.5, 0.7)$ → normalizada $(0.58, 0.81)$: su vecino más cercano ya es un documento de «git» (coseno ≈ 1.00 con $(0.6, 0.8)$ frente a 0.95 con $(0.8, 0.6)$, el «cache» más cercano). Es la aritmética de vectores de word2vec (rey − hombre + mujer) aplicada a una consulta, y el «*boost* por categoría» de un recomendador hecho en el espacio de embeddings.

In [ ]:
topics = sorted(set(topic)); mu = {t: En[(topic == t) & (lang == "es")].mean(0) for t in topics}
v_cache_git = mu["git"] - mu["cache"]
qs = np.where((topic == "cache") & (lang == "es"))[0][:100]; alphas = [0, .25, .5, .75, 1, 1.25, 1.5, 2, 3]
curva = {"cache": [], "git": [], "otros": []}
for a in alphas:
    q = En[qs] + a * v_cache_git; q /= np.linalg.norm(q, axis=1, keepdims=True)
    sc = q @ En.T; sc[np.arange(len(qs)), qs] = -9; top = np.argsort(-sc, axis=1)[:, :10]
    fc, fg = float((topic[top] == "cache").mean()), float((topic[top] == "git").mean()); curva["cache"].append(fc); curva["git"].append(fg); curva["otros"].append(1 - fc - fg)
fig, ax = plt.subplots(figsize=(7, 3.6))
for k, c in (("cache", AZUL), ("git", NARANJA), ("otros", GRIS)): ax.plot(alphas, curva[k], "o-", color=c, label=f"% de resultados de «{k}»")
ax.set_xlabel("intensidad del steering α  (q' = q + α·v_{cache→git})"); ax.set_ylabel("fracción del top-10"); ax.legend(frameon=False); ax.set_title("Mover la consulta en la dirección «git − cache» cambia los resultados"); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** La transición es **suave y casi monótona**: con α≈0.5 todavía domina «cache», hacia α≈1 hay mezcla y con α ≥ 1.5 todo el top-10 es «git». Una dirección calculada con *medias* (sin entrenamiento) basta para controlar el tema del retrieval en este juguete.
> 🟡 En embedders reales funciona de forma menos limpia (conceptos entrelazados, direcciones que cambian de capa a capa, normas); además *AxBench* (Wu et al., 2025) encontró que, para steering de LLMs, **líneas base simples** (prompting, diferencia de medias) **superan a los SAE**. Por eso conviene comparar siempre contra la diferencia de medias antes de afirmar que una *feature* aprendida «controla» algo.

> ### 🎬 GEMINI-VIDEO · ID: `M17-VID-1` · ⚠️ REEMPLAZAR ESTA CELDA
> **Steering: sumar un vector de concepto desplaza la consulta y sus resultados**
>
> **Qué muestra:** Animación en 2D de una consulta que se desplaza por el vector «git − cache» y pasa del clúster de documentos de caché al de git, con el top-3 cambiando.
> **Formato:** MP4 16:9, 1080p | Duración: 28 s | **Texto alternativo:** Una consulta se desplaza por la dirección que va de la nube de caché a la de git y su top-3 cambia de caché a git.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Animación técnica limpia, 16:9, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif, ritmo calmado, sin voz en off, sin música. Textos en español.
> Escenas:
> 0–6 s: un plano 2D sin ejes numéricos con dos nubes de puntos: una azul etiquetada «documentos de caché» (izquierda) y una naranja etiquetada «documentos de git» (derecha), y una estrella negra «consulta q» dentro de la nube azul. Un panel a la derecha lista «Top-3: cache, cache, cache» con barras azules.
> 6–12 s: aparecen los centroides de cada nube como puntos grandes y una flecha gris gruesa del centroide azul al naranja con la etiqueta «v = media(git) − media(cache)».
> 12–22 s: la estrella se desplaza gradualmente a lo largo de una flecha verde paralela a v; sobre ella un contador «α = 0 → 0.5 → 1 → 1.5». Los tres vecinos más cercanos se resaltan con líneas finas y el panel de la derecha cambia: «cache, cache, git» (α = 1), luego «git, git, git» (α = 1.5), con barras que pasan de azul a naranja.
> 22–28 s: la estrella queda dentro de la nube naranja; texto final «Una dirección de concepto controla el retrieval (¡mídelo y compáralo con baselines!)».
> Evita: voz en off, música, texto ilegible, números inventados, efectos 3D, transiciones bruscas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M17-VID-1.mp4` en la carpeta del módulo y reemplaza esta celda por: `<video src="assets/M17-VID-1.mp4" controls width="100%"></video>`
>
> **Mientras tanto (placeholder):**
> ```text
> [nube cache]  q ★ ------ α·v ------> ★ [nube git]
> top-3: cache cache cache  ->  cache cache git  ->  git git git
> ```

In [ ]:
# Borrado lineal de un concepto (idioma) con la dirección de diferencia de medias: la versión más simple de LEACE (Belrose et al. 2023)
u = En[lang == "en"].mean(0) - En[lang == "es"].mean(0); u /= np.linalg.norm(u)
Ep = En - np.outer(En @ u, u); Epn = Ep / np.linalg.norm(Ep, axis=1, keepdims=True); y_en = (lang == "en").astype(int)
acc_antes, acc_despues = sonda(En, y_en), sonda(Ep, y_en)
def stats_retrieval(M):
    qi = np.where(lang == "es")[0][:200]; sc = M[qi] @ M.T; sc[np.arange(len(qi)), qi] = -9; top = np.argsort(-sc, axis=1)[:, :10]
    return float((lang[top] == "es").mean()), float((facet[top] == facet[qi][:, None]).mean())
mis_l0, mis_f0 = stats_retrieval(En); mis_l1, mis_f1 = stats_retrieval(Epn)
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.4))
ax[0].bar(["antes", "tras borrar la dirección"], [acc_antes, acc_despues], color=[AZUL, VERDE]); ax[0].axhline(.5, color=GRIS, ls="--"); ax[0].set_ylim(0, 1.05); ax[0].set_title("Sonda lineal de IDIOMA"); ax[0].set_ylabel("precisión")
w = .38; ax[1].bar(np.arange(2) - w/2, [mis_l0, mis_f0], w, color=AZUL, label="antes"); ax[1].bar(np.arange(2) + w/2, [mis_l1, mis_f1], w, color=VERDE, label="tras borrar")
ax[1].set_xticks(range(2)); ax[1].set_xticklabels(["top-10 en el mismo idioma", "top-10 de la misma faceta"]); ax[1].legend(frameon=False); ax[1].set_ylim(0, 1.05); ax[1].set_title("Efecto en el retrieval (consultas ES)")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Tras proyectar fuera una sola dirección, la sonda lineal de idioma cae al **azar** (≈ 0.5): la información ya no es *linealmente* legible. Pero el **retrieval no cambia**: sigue devolviendo documentos del mismo idioma. Lección de interpretabilidad: **borrar lo que un probe lee no cambia necesariamente lo que el sistema hace** (la información de idioma está también en *qué palabras* aparecen, redundante en muchas dimensiones). Para borrado óptimo se usa **LEACE** (Belrose et al., 2023, borrado lineal en forma cerrada); sigue habiendo información no lineal.

## 5. Dimensiones dominantes, outliers y anisotropía
**Timkey & van Schijndel (EMNLP 2021):** en BERT y GPT-2, las similitudes (coseno, euclídea) están dominadas por **1–3 «rogue dimensions»** de varianza enorme, que además no coinciden con las que importan para el comportamiento; estandarizar las dimensiones revela la calidad real de la representación. **Kovaleva et al. (2021)** encuentran *outlier dimensions* (escalas y sesgos de LayerNorm de gran magnitud) y muestran que desactivarlas degrada mucho el modelo: **son «importantes» para el modelo pero distorsionan la similitud**. Ethayarajh (2019) documenta la anisotropía de las representaciones contextuales.
Nuestro LSI **no tiene** esas dimensiones (no es un transformer); **simulamos** una (inyectamos una dimensión con desviación ≈ 100× la típica) para ver el mecanismo; con un embedder real, `diagnostico_dimensiones(E)` te dirá si las tienes.

In [ ]:
def con_dim_rogue(E, escala=100.0, seed=0):
    r = np.random.default_rng(seed); out = E.copy(); out[:, 3] = E[:, 3] * 1.0 + escala * E.std() * (1.0 + 0.3 * r.standard_normal(len(E))); return out
E_rogue = con_dim_rogue(E)
def prec_facet(M, n_q=300):
    M = M / np.linalg.norm(M, axis=1, keepdims=True); qi = np.random.default_rng(1).choice(len(M), n_q, replace=False); sc = M[qi] @ M.T; sc[np.arange(n_q), qi] = -9
    return float((facet[np.argsort(-sc, axis=1)[:, :10]] == facet[qi][:, None]).mean())
Z_std = (E_rogue - E_rogue.mean(0)) / (E_rogue.std(0) + 1e-12)
res = {"LSI limpio": diagnostico_dimensiones(E), "con dimensión rogue (simulada)": diagnostico_dimensiones(E_rogue)}
res["LSI limpio"]["precisión@10 de faceta"] = prec_facet(E); res["con dimensión rogue (simulada)"]["precisión@10 de faceta"] = prec_facet(E_rogue)
def falsos_umbral(M, u=0.5):
    M = M / np.linalg.norm(M, axis=1, keepdims=True); a, b = np.random.default_rng(3).integers(0, len(M), (2, 20000)); m = (a != b) & (facet[a] != facet[b]); return float((np.sum(M[a] * M[b], 1)[m] > u).mean())
res["LSI limpio"]["pares NO relacionados con coseno > 0.5"] = falsos_umbral(E); res["con dimensión rogue (simulada)"]["pares NO relacionados con coseno > 0.5"] = falsos_umbral(E_rogue)
d_rogue = pd.DataFrame(res).T; d_rogue["tras estandarizar: precisión@10"] = [prec_facet((E - E.mean(0)) / (E.std(0) + 1e-12)), prec_facet(Z_std)]; display(d_rogue.round(3))
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5)); v = np.sort(E_rogue.var(0))[::-1]; ax[0].bar(range(len(v)), v, color=AZUL); ax[0].set_title("Varianza por dimensión (con rogue)"); ax[0].set_xlabel("dimensiones ordenadas"); ax[0].set_ylabel("varianza")
ax[1].bar(["limpio", "rogue", "rogue\nestandarizado"], [d_rogue.loc["LSI limpio", "coseno_medio"], d_rogue.loc["con dimensión rogue (simulada)", "coseno_medio"], d_rogue.loc["con dimensión rogue (simulada)", "coseno_medio_estandarizado"]], color=[VERDE, ROJO, AZUL]); ax[1].set_title("Coseno medio entre textos al azar"); ax[1].set_ylabel("coseno medio")
ax[2].bar(["limpio", "rogue"], [d_rogue.loc["LSI limpio", "pares NO relacionados con coseno > 0.5"], d_rogue.loc["con dimensión rogue (simulada)", "pares NO relacionados con coseno > 0.5"]], color=[VERDE, ROJO]); ax[2].set_title("Pares no relacionados que pasan un umbral de coseno 0.5"); ax[2].set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Con **una sola** dimensión dominante: (1) concentra ≈ 94 % de la varianza; (2) el **coseno medio entre textos al azar se dispara** (≈ 1: anisotropía extrema); (3) **casi todos los pares no relacionados superan un umbral de coseno 0.5**: cualquier regla del tipo «considerar duplicado/relevante si coseno > u», filtros de similitud, *clustering* o diversificación (MMR) **se rompe**; (4) sin embargo la **precisión@10 del ranking apenas cambia** en este juguete (ver columna `precisión@10 de faceta`): el ranking depende de diferencias *relativas*, no del nivel absoluto del coseno. **Estandarizar** cada dimensión devuelve el coseno medio a ≈ 0. Es el diagnóstico y remedio de Timkey & van Schijndel (ellos reportan efectos en medidas de similitud y en la calidad aparente de la representación).
> Nota de honestidad: la dimensión es **simulada** (no es una medición de ningún modelo real). Ejecuta `diagnostico_dimensiones` sobre tu embedder real: si `var_top1` es grande o el coseno medio entre textos no relacionados es alto, aplica *whitening* o estandarización (con un sistema de producción, **valida el efecto en nDCG**, módulo 15).

### 5b. 🧪 Hubness: el documento que aparece en todas las búsquedas
Otra patología geométrica con consecuencias directas en retrieval: en alta dimensión, unos pocos puntos (**hubs**) aparecen entre los $k$ vecinos más cercanos de muchísimos otros, y muchos puntos (**anti-hubs**) no aparecen en ninguno (Radovanović, Nanopoulos & Ivanović, JMLR 2010, *Hubs in Space*: la asimetría de la distribución de la *k-ocurrencia* $N_k$ crece con la dimensión intrínseca). En un buscador, un hub es «el documento que sale para todo» y un anti-hub es un documento que nadie encuentra. Remedios clásicos: **centrar** (restar la media, que también quita la anisotropía de §5) y **CSLS** (Conneau et al., ICLR 2018), que penaliza a cada documento por su similitud media con sus vecinos: $\mathrm{CSLS}(q,d)=2\cos(q,d)-r(q)-r(d)$; para ordenar documentos de una query fija solo importa $-r(d)$.
Medimos $N_{10}$ con los documentos como queries (lo estándar en esta literatura) y la precisión@10 de faceta, con las mismas funciones de §5.

In [ ]:
from scipy.stats import skew
def k_ocurrencia(M, k=10, csls=False):
    M = M / np.linalg.norm(M, axis=1, keepdims=True); S = M @ M.T; np.fill_diagonal(S, -9)
    if csls:
        r = np.sort(S, axis=1)[:, -k:].mean(1); S = S - r[None, :]          # penaliza documentos «populares»
    top = np.argpartition(-S, k, axis=1)[:, :k]
    return np.bincount(top.ravel(), minlength=len(M)), top
variantes = {"LSI-64": E, "LSI-64 con rogue": E_rogue, "rogue + centrado": E_rogue - E_rogue.mean(0), "rogue + CSLS": E_rogue}
filas_h, Nk_all = [], {}
for nombre, M in variantes.items():
    Nk, top = k_ocurrencia(M, csls=nombre.endswith("CSLS")); Nk_all[nombre] = Nk
    filas_h.append({"variante": nombre, "asimetría de N10": skew(Nk), "máx N10": int(Nk.max()), "% anti-hubs (N10 = 0)": 100 * (Nk == 0).mean(),
                    "% apariciones del 1 % top": 100 * np.sort(Nk)[::-1][: max(1, len(Nk) // 100)].sum() / Nk.sum(),
                    "precisión@10 de faceta": float((facet[top] == facet[:, None]).mean())})
d_hub = pd.DataFrame(filas_h).set_index("variante"); display(d_hub.round(3))
fig, ax = plt.subplots(figsize=(8, 3.6))
for (nombre, Nk), c in zip(Nk_all.items(), [AZUL, ROJO, VERDE, NARANJA]):
    ax.hist(Nk, bins=np.arange(0, max(v.max() for v in Nk_all.values()) + 2, 2), histtype="step", lw=2, color=c, label=nombre)
ax.set_yscale("log"); ax.set_xlabel("$N_{10}$: en cuántas listas top-10 aparece cada documento"); ax.set_ylabel("nº de documentos (log)")
ax.set_title("Distribución de la k-ocurrencia (hubness)"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** La media de $N_{10}$ es siempre 10 (cada lista tiene 10 huecos); lo que cambia es la **cola**. Con la dimensión dominante la distribución se estira: unos pocos documentos acaparan muchas apariciones y crece el porcentaje de anti-hubs, a la vez que cae la precisión de faceta. Centrar y CSLS acortan la cola (comprueba en la tabla cuánto de la precisión recuperan). En producción es una señal **sin etiquetas** barata: la concentración del top-1 sobre el tráfico (módulo 18 §9) y la lista de documentos que nunca aparecen en ningún top-k delatan hubs y anti-hubs antes de que un usuario se queje de que «siempre sale el mismo».

## 6. Superposición y sparse autoencoders (SAE)

### 6.1 Por qué los ejes no significan nada: superposición
Un espacio de $d$ dimensiones solo tiene $d$ direcciones ortogonales, pero un modelo quiere representar **muchos más** conceptos ($n\gg d$). **Elhage et al. (Anthropic, 2022, *Toy Models of Superposition*)** muestran que, si los conceptos son **dispersos** (rara vez activos a la vez), el modelo los almacena en direcciones **casi ortogonales** que se interfieren poco: *superposición*. Resultado: una neurona/dimensión mezcla varios conceptos (*polisemantismo*), y por eso mirar ejes individuales no funciona.
Reproduzcamos el juguete: 5 características en 2 dimensiones, variando la **dispersión** $S$ (probabilidad de que una característica esté apagada).

In [ ]:
import torch.nn as nn, torch.nn.functional as F
def juguete_superposicion(S, n=5, m=2, pasos=2500, seed=0):
    """Entrena x_hat = ReLU(W^T W x + b) con entradas dispersas (cada una apagada con prob. S). W: (m, n) -> columnas = direcciones."""
    torch.manual_seed(seed); W = nn.Parameter(torch.randn(m, n) * .3); b = nn.Parameter(torch.zeros(n)); opt = torch.optim.Adam([W, b], lr=5e-3)
    for _ in range(pasos):
        x = torch.rand(1024, n) * (torch.rand(1024, n) > S); xh = F.relu((x @ W.T) @ W + b); loss = ((x - xh) ** 2).mean() * n
        opt.zero_grad(); loss.backward(); opt.step()
    return W.detach().numpy(), loss.item()
fig, ax = plt.subplots(1, 4, figsize=(14, 3.5)); cols5 = [AZUL, NARANJA, VERDE, ROJO, GRIS]
for a, S in zip(ax, (0.0, 0.7, 0.9, 0.97)):
    W, l = juguete_superposicion(S)
    for j in range(5): a.annotate("", xy=W[:, j], xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=cols5[j], lw=2)); a.text(*(W[:, j] * 1.12), f"f{j+1}", color=cols5[j], fontsize=9)
    a.set_xlim(-1.4, 1.4); a.set_ylim(-1.4, 1.4); a.set_aspect("equal"); a.set_title(f"dispersión S={S}\n(normas: {np.linalg.norm(W, axis=0).round(1)})", fontsize=8)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Con características **densas** (S=0) el modelo solo representa bien ~2 de las 5 (normas de longitud muy desigual: las demás se sacrifican). Al aumentar la dispersión, las **cinco** flechas aparecen con norma ≈ 1 y se reparten por el plano como un **pentágono**: 5 características en 2 dimensiones. *Superposición*: más conceptos que ejes, pagando interferencia.

> ### 🎨 GEMINI-IMAGEN · ID: `M17-IMG-1` · ⚠️ REEMPLAZAR ESTA CELDA
> **Superposición: cinco características en dos dimensiones**
>
> **Qué muestra:** Imagen conceptual de la geometría de la superposición: con características dispersas, cinco direcciones forman un pentágono en un plano de 2 dimensiones; con características densas solo caben dos.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Dos planos: con características densas solo caben dos direcciones; con características dispersas cinco direcciones forman un pentágono.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif legible, flechas claras, sin ilustraciones decorativas. 16:9.
> Dos paneles lado a lado, cada uno un plano 2D con un círculo gris fino de radio 1 y ejes grises sin números.
> Panel izquierdo, título «Características DENSAS (S = 0)»: solo dos flechas largas, azul «f1» y naranja «f2», perpendiculares; tres flechas muy cortas casi invisibles en gris «f3, f4, f5». Debajo: «Solo 2 características caben bien».
> Panel derecho, título «Características DISPERSAS (S = 0.97)»: cinco flechas de igual longitud desde el origen formando un pentágono regular, con colores azul «f1», naranja «f2», verde «f3», rojo apagado «f4», gris «f5», y líneas finas punteadas uniendo las puntas. Debajo: «5 características en 2 dimensiones: superposición».
> Entre los dos paneles una flecha gris con el texto «más dispersión».
> Evita: texto ilegible o inventado, ilustraciones decorativas, sombras 3D, más de cinco flechas, ejes con números.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M17-IMG-1.png` en la carpeta del módulo y reemplaza esta celda por: `![Dos planos: con características densas solo caben dos direcciones; con características dispersas cinco direcciones forman un pentágono.](assets/M17-IMG-1.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> S=0:    f1 ──►  f2 │  (3 casi sin norma)         S=0.97:  pentágono: f1..f5 con |f|≈1 en 2D
> ```

> 👀 **Qué debes observar:** a la izquierda (densas) solo dos flechas largas, ortogonales: el modelo sacrifica tres características. A la derecha (dispersas) cinco flechas en pentágono: caben más conceptos que dimensiones a cambio de **interferencia** (coseno 0.31 entre flechas vecinas y −0.81 entre no vecinas; la ReLU y el sesgo absorben la interferencia negativa). Conclusión para embeddings: un eje del vector no tiene por qué ser un concepto; buscar «la neurona del español» puede no tener sentido, y por eso se entrena un diccionario (6.2). Compáralo con la figura de código de arriba.

### 6.2 Sparse autoencoders: deshacer la superposición
Si los conceptos están en superposición, buscamos un **diccionario sobrecompleto** $\{a_j\}_{j=1}^{m}$ ($m>d$) tal que cada vector se reconstruya con **pocos átomos**:
$$\hat x = W_{dec}\,z + b_{dec},\qquad z=\text{codificación dispersa}(x)$$
- **ReLU + L1** (Bricken et al., Anthropic 2023; Cunningham et al. 2023): $z=\mathrm{ReLU}(W_{enc}(x-b_{dec})+b_{enc})$ y pérdida $\lVert x-\hat x\rVert^2+\lambda\lVert z\rVert_1$. Hay que ajustar $\lambda$ para el $L_0$ deseado y la penalización L1 **encoge** los valores (sesgo).
- **TopK** (Gao et al., OpenAI 2024; ya en el *k-sparse autoencoder* de Makhzani & Frey, 2013): $z=\mathrm{TopK}(W_{enc}(x-b_{dec})+b_{enc})$ conserva solo las $k$ mayores activaciones. **$L_0=k$ exacto**, sin $\lambda$, mejor frontera reconstrucción–dispersión y pocos latentes muertos con inicialización y pérdida auxiliar adecuadas.
- **AuxK** (Gao et al.): los latentes *muertos* (sin activarse en mucho tiempo) intentan reconstruir el **residuo** con sus top-$k_{aux}$ preactivaciones; resucita átomos.
- Columnas de $W_{dec}$ con **norma 1** (si no, el modelo «hace trampa» escalando).
Medimos **NMSE** $=\lVert X-\hat X\rVert^2/\lVert X-\bar X\rVert^2$. (JumpReLU, Rajamanoharan et al. 2024, es otra variante que aprende un umbral por latente; no la implementamos.)

> ### 🎨 GEMINI-IMAGEN · ID: `M17-IMG-2` · ⚠️ REEMPLAZAR ESTA CELDA
> **Arquitectura de un sparse autoencoder TopK**
>
> **Qué muestra:** Flujo x → encoder lineal → TopK (solo k activos) → decoder con átomos unitarios → x̂, con pérdida de reconstrucción, y la interpretación de cada átomo como una característica.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Un embedding pasa por un encoder lineal, una activación TopK que deja pocos latentes activos y un decoder de átomos unitarios que reconstruye el vector.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif legible, flechas claras, sin ilustraciones decorativas. 16:9.
> De izquierda a derecha cinco bloques conectados por flechas:
> 1) Caja azul «x ∈ R^64 (embedding del documento)» dibujada como una fila de 12 celdas.
> 2) Caja azul «Encoder lineal W_enc (x − b_dec) + b_enc → 256 preactivaciones» dibujada como una fila larga de 24 celdas grises.
> 3) Caja naranja «TopK (k = 4)»: de la fila larga solo 4 celdas quedan en naranja intenso con valores «0.9, 0.6, 0.4, 0.3» y el resto tachado en gris; etiqueta «z: dispersa, L0 = k».
> 4) Caja verde «Decoder W_dec (256 átomos de norma 1) z + b_dec» con un pequeño abanico de 4 flechas verdes que se suman.
> 5) Caja azul «x̂ ≈ x» con una flecha de retorno punteada gris hacia 1 con el texto «pérdida: ‖x − x̂‖² / varianza».
> Debajo del bloque 4, un recuadro gris: «Cada átomo = una característica: se interpreta por sus documentos más activos y por sus términos». Un recuadro más pequeño naranja: «AuxK: los latentes muertos reconstruyen el residuo».
> Título superior: «Sparse autoencoder TopK sobre embeddings».
> Evita: texto ilegible o inventado, ilustraciones decorativas, sombras 3D, más de cinco bloques, flechas ambiguas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M17-IMG-2.png` en la carpeta del módulo y reemplaza esta celda por: `![Un embedding pasa por un encoder lineal, una activación TopK que deja pocos latentes activos y un decoder de átomos unitarios que reconstruye el vector.](assets/M17-IMG-2.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> x(64) -> [W_enc] -> 256 preact -> [TopK k=4] -> z dispersa -> [W_dec, átomos |a|=1] -> x_hat
>                                               pérdida NMSE (+ AuxK en latentes muertos)
> ```

> 👀 **Qué debes observar:** el cuello no es estrecho sino **ancho** ($m > d$, sobrecompleto) y lo que obliga a separar conceptos es el TopK: solo $k$ átomos activos por vector. Cada columna del decoder (unitaria) es una característica candidata; la pérdida es solo de reconstrucción, así que la interpretabilidad **no** está garantizada: se mide (sección 7 y proyecto). Es un *dictionary learning* clásico, pariente del codebook VQ del módulo 08 (allí 1 átomo por vector; aquí $k$).

### 6.3 Implementación desde cero (PyTorch)
TopK SAE con inicialización *tied*, `b_dec` = media de los datos, decoder normalizado en cada paso y AuxK; más el SAE ReLU+L1 para comparar.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class TopKSAE(nn.Module):
    """SAE con activación TopK (Gao et al., OpenAI 2024; k-sparse AE de Makhzani & Frey 2013).
    z = TopK(W_enc (x - b_dec) + b_enc) ; x_hat = W_dec z + b_dec ; columnas de W_dec de norma 1. L0 = k exacto (sin λ que ajustar)."""
    def __init__(self, d: int, n_lat: int, k: int):
        super().__init__(); self.d, self.n, self.k = d, n_lat, k
        W = F.normalize(torch.randn(d, n_lat), dim=0)
        self.W_dec = nn.Parameter(W.clone()); self.W_enc = nn.Parameter(W.clone().T.contiguous())      # inicialización "tied" (transpuesta)
        self.b_enc = nn.Parameter(torch.zeros(n_lat)); self.b_dec = nn.Parameter(torch.zeros(d))
        self.register_buffer("pasos_sin_activar", torch.zeros(n_lat))
    def pre(self, x): return (x - self.b_dec) @ self.W_enc.T + self.b_enc
    def encode(self, x, k=None):
        pre = self.pre(x); v, i = pre.topk(k or self.k, dim=-1)
        return torch.zeros_like(pre).scatter(-1, i, F.relu(v)), pre
    def decode(self, z): return z @ self.W_dec.T + self.b_dec
    def forward(self, x): z, pre = self.encode(x); return self.decode(z), z, pre
    @torch.no_grad()
    def normaliza_decoder(self): self.W_dec.data = F.normalize(self.W_dec.data, dim=0)

def entrena_topk(X, n_lat=256, k=4, pasos=1500, bs=256, lr=2e-3, auxk=True, k_aux=None, ventana_muerto=200, seed=0):
    """Entrena un TopK SAE. auxk=True añade la pérdida AuxK de Gao et al.: los latentes 'muertos' (sin activarse en `ventana_muerto`
    pasos) reconstruyen el RESIDUO del batch con sus top-k_aux preactivaciones -> los resucita. Devuelve (sae, historial)."""
    torch.manual_seed(seed); X = torch.as_tensor(X, dtype=torch.float32); d = X.shape[1]
    sae = TopKSAE(d, n_lat, k); sae.b_dec.data = X.mean(0).clone()                   # b_dec = media de los datos (centra la reconstrucción)
    opt = torch.optim.Adam(sae.parameters(), lr=lr); k_aux = k_aux or min(d // 2, n_lat // 4); var = ((X - X.mean(0)) ** 2).sum(1).mean()
    hist = {"nmse": [], "muertos": []}
    for t in range(pasos):
        x = X[torch.randint(0, len(X), (bs,))]; xh, z, pre = sae(x); err = x - xh
        loss = (err ** 2).sum(1).mean() / var
        activos = (z > 0).any(0); sae.pasos_sin_activar += 1; sae.pasos_sin_activar[activos] = 0
        muertos = sae.pasos_sin_activar > ventana_muerto
        if auxk and muertos.any():
            pm = pre.masked_fill(~muertos, -1e9); v, i = pm.topk(min(k_aux, int(muertos.sum())), dim=-1)
            za = torch.zeros_like(pre).scatter(-1, i, F.relu(v)); e_hat = za @ sae.W_dec.T
            loss = loss + (1 / 32) * ((err.detach() - e_hat) ** 2).sum(1).mean() / var
        opt.zero_grad(); loss.backward(); opt.step(); sae.normaliza_decoder()
        if t % 25 == 0: hist["nmse"].append(float(((err.detach()) ** 2).sum(1).mean() / var)); hist["muertos"].append(int(muertos.sum()))
    return sae.eval(), hist

def entrena_relu_l1(X, n_lat=256, lam=1e-2, pasos=1500, bs=256, lr=2e-3, seed=0):
    """SAE 'clásico' (Bricken et al., Anthropic 2023; Cunningham et al. 2023): z = ReLU(W_enc(x-b)+b_enc), pérdida = NMSE + λ·||z||_1."""
    torch.manual_seed(seed); X = torch.as_tensor(X, dtype=torch.float32); d = X.shape[1]
    sae = TopKSAE(d, n_lat, k=n_lat); sae.b_dec.data = X.mean(0).clone(); opt = torch.optim.Adam(sae.parameters(), lr=lr); var = ((X - X.mean(0)) ** 2).sum(1).mean()
    for _ in range(pasos):
        x = X[torch.randint(0, len(X), (bs,))]; z = F.relu(sae.pre(x)); xh = sae.decode(z)
        loss = ((x - xh) ** 2).sum(1).mean() / var + lam * z.abs().sum(1).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sae.normaliza_decoder()
    sae.relu_mode = True; return sae.eval()

@torch.no_grad()
def codifica(sae, X, k=None):
    """Latentes (n, n_lat) y reconstrucción. Si el SAE es ReLU+L1 usa ReLU sin TopK."""
    X = torch.as_tensor(X, dtype=torch.float32)
    z = F.relu(sae.pre(X)) if getattr(sae, "relu_mode", False) else sae.encode(X, k)[0]
    return z, sae.decode(z)

def nmse(X, Xh):
    X, Xh = torch.as_tensor(X, dtype=torch.float32), torch.as_tensor(Xh, dtype=torch.float32)
    return float(((X - Xh) ** 2).sum() / ((X - X.mean(0)) ** 2).sum())

### 6.4 ¿Recupera características *conocidas*? (verdad terreno)
Antes de interpretar *features* de un embedding real necesitamos saber si el método **recupera lo que hay**. Generamos datos con **64 direcciones reales** en 24 dimensiones (superposición: $64>24$), cada muestra mezcla **3** al azar. Medimos, para cada dirección real, el coseno con el mejor átomo aprendido.

In [ ]:
def datos_plantados(n_true=64, d=24, k_act=3, n=6000, seed=0):
    g = torch.Generator().manual_seed(seed); Dtrue = F.normalize(torch.randn(n_true, d, generator=g), dim=1)
    idx = torch.stack([torch.randperm(n_true, generator=g)[:k_act] for _ in range(n)]); coef = torch.rand(n, k_act, generator=g) * 1.5 + .5
    return torch.zeros(n, n_true).scatter(1, idx, coef) @ Dtrue + 0.01 * torch.randn(n, d, generator=g), Dtrue
Xp, Dtrue = datos_plantados()
from sklearn.decomposition import PCA
pasos = 2500 if FAST_DEV_RUN else 6000; resultados = {}
s_tk, _ = entrena_topk(Xp.numpy(), n_lat=128, k=3, pasos=pasos); zt, xt = codifica(s_tk, Xp); resultados["TopK (k=3)"] = (s_tk.W_dec.data.T, nmse(Xp, xt), float((zt > 0).float().sum(1).mean()))
for lam in (3e-2, 1e-1):
    s_l1 = entrena_relu_l1(Xp.numpy(), 128, lam, pasos); zl, xl = codifica(s_l1, Xp); resultados[f"ReLU+L1 (λ={lam})"] = (s_l1.W_dec.data.T, nmse(Xp, xl), float((zl > 0).float().sum(1).mean()))
pca = PCA(24).fit(Xp.numpy()); resultados["PCA (24 comp.)"] = (torch.tensor(pca.components_), nmse(Xp, torch.tensor(pca.inverse_transform(pca.transform(Xp.numpy())))), 24.0)
filas = []
for n, (W, e, l0) in resultados.items():
    mejor = (W @ Dtrue.T).abs().max(0).values; filas.append({"método": n, "características recuperadas (cos>0.9)": float((mejor > .9).float().mean()), "cos medio con el mejor átomo": float(mejor.mean()), "NMSE": e, "L0": l0})
rec = pd.DataFrame(filas).set_index("método"); display(rec.round(3))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5)); rec["características recuperadas (cos>0.9)"].plot(kind="bar", ax=ax[0], color=[VERDE, AZUL, AZUL, GRIS], rot=15); ax[0].set_ylabel("fracción de las 64 recuperadas"); ax[0].set_title("Recuperación de características plantadas")
for n, c in zip(rec.index, [VERDE, AZUL, AZUL, GRIS]): ax[1].scatter(rec.loc[n, "L0"], rec.loc[n, "NMSE"], s=70, color=c); ax[1].annotate(n, (rec.loc[n, "L0"], rec.loc[n, "NMSE"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
ax[1].set_xscale("log"); ax[1].set_xlabel("L0 (átomos activos por muestra, log)"); ax[1].set_ylabel("NMSE"); ax[1].set_title("Frontera reconstrucción–dispersión"); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) **PCA no recupera ninguna** característica (no puede: solo hay 24 ejes y 64 conceptos). (2) TopK con $k=3$ (el $L_0$ verdadero) recupera casi todas con **NMSE bajo**. (3) ReLU+L1 con $\lambda$ pequeño es denso y **no recupera**; con $\lambda$ mayor recupera pero a costa de **NMSE más alto** (el encogimiento L1) y $L_0$ difícil de fijar. Es el resultado cualitativo que motivó TopK en Gao et al. 2024. ⚠️ Es un problema **sintético y fácil** (ortogonalidad casi perfecta, dispersión exacta): en datos reales hay mucho menos margen.

### 6.5 Frontera reconstrucción–dispersión y latentes muertos en los embeddings de Políglota

In [ ]:
pasos = 1500 if FAST_DEV_RUN else 4000; ks = [2, 4, 8, 16]; fr_topk, fr_l1 = [], []
for k in ks:
    s, _ = entrena_topk(E, n_lat=96, k=k, pasos=pasos // 2); z, xh = codifica(s, E); fr_topk.append((k, nmse(E, xh)))
for lam in (3e-3, 1e-2, 3e-2, 1e-1):
    s = entrena_relu_l1(E, 96, lam, pasos // 2); z, xh = codifica(s, E); fr_l1.append((float((z > 0).float().sum(1).mean()), nmse(E, xh)))
hist_aux = {a: entrena_topk(E, n_lat=96, k=4, pasos=pasos, lr=8e-3, auxk=a)[1] for a in (False, True)}
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.6))
ax[0].plot(*zip(*fr_topk), "o-", color=VERDE, label="TopK"); ax[0].plot(*zip(*fr_l1), "s-", color=AZUL, label="ReLU + L1 (variando λ)"); ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_xlabel("L0 medio (log)"); ax[0].set_ylabel("NMSE (log)"); ax[0].legend(frameon=False); ax[0].set_title("Frontera sobre embeddings de Políglota (96 latentes)")
for a, c in ((False, ROJO), (True, VERDE)): ax[1].plot(np.arange(len(hist_aux[a]["muertos"])) * 25, hist_aux[a]["muertos"], color=c, label="con AuxK" if a else "sin AuxK")
ax[1].set_xlabel("paso de entrenamiento"); ax[1].set_ylabel("latentes muertos (sin activarse en 200 pasos)"); ax[1].legend(frameon=False); ax[1].set_title("Latentes muertos (lr alto = 8e-3)")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) A igual $L_0$, TopK reconstruye mejor (curva por debajo) y no necesita ajustar $\lambda$. (2) Los latentes muertos aparecen con $lr$ alto; **AuxK no siempre los evita en un problema pequeño** como este (los datos solo tienen 36 facetas: sobran átomos): la mitigación efectiva depende de tamaño del diccionario, datos y *lr*. En los LLM de Gao et al., con el AuxK y la inicialización adecuada, reportan muy pocos latentes muertos incluso a gran escala.

## 7. Un SAE sobre los embeddings de Políglota: ¿son interpretables sus *features*?
Entrenamos un TopK SAE ($m=96$ latentes, $k=4$) sobre LSI-64. Interpretamos una *feature* de dos maneras: (a) sus **documentos más activos** y (b) como el embedding LSI es lineal en los términos, **proyectamos su átomo del decoder al espacio de términos** ($a^\top V_k$, con $V_k$ los componentes SVD) para leer sus **palabras** — algo imposible con embedders de transformer (ahí solo (a)).
**Medida cuantitativa de interpretabilidad (con verdad terreno):** *pureza* = fracción de los 20 documentos más activos de una *feature* que comparten la **misma faceta**; comparamos con las direcciones de **k-means** y de **PCA**.

In [ ]:
pasos = 2000 if FAST_DEV_RUN else 5000
sae, hist = entrena_topk(E, n_lat=96, k=4, pasos=pasos, auxk=True); Z, Xh = codifica(sae, E); Z = Z.numpy()
activos = (Z > 0).sum(0) >= 10
def pureza(scores, etiquetas, top=20): return np.array([np.bincount(etiquetas[np.argsort(-scores[:, j])[:top]]).max() / top for j in range(scores.shape[1])])
from sklearn.cluster import KMeans
n_f = int(activos.sum()); km = KMeans(n_f, n_init=3, random_state=0).fit(E); Ck = km.cluster_centers_; sc_km = E @ Ck.T / (np.linalg.norm(Ck, axis=1) + 1e-9)
pc = PCA(min(n_f, 64)).fit(E); sc_pc = np.abs((E - pc.mean_) @ pc.components_.T)
P = {"SAE TopK": pureza(Z[:, activos], fac_id), "k-means": pureza(sc_km, fac_id), "PCA": pureza(sc_pc, fac_id)}
print(f"SAE: {n_f} latentes activos de 96 · NMSE={nmse(E, Xh):.3f} · L0=4 · facetas cubiertas por alguna feature con pureza ≥ 0.8: "
      f"{len({np.bincount(fac_id[np.argsort(-Z[:, j])[:20]]).argmax() for j in np.where(activos)[0] if P['SAE TopK'][list(np.where(activos)[0]).index(j)] >= .8})}/36")
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.5)); ax[0].bar(list(P), [v.mean() for v in P.values()], color=[VERDE, AZUL, GRIS]); ax[0].set_ylabel("pureza media (misma faceta en el top-20)"); ax[0].set_title("Interpretabilidad medida")
for (n, v), c in zip(P.items(), (VERDE, AZUL, GRIS)): ax[1].hist(v, bins=np.linspace(0, 1, 11), alpha=.55, color=c, label=n)
ax[1].set_xlabel("pureza por feature"); ax[1].set_ylabel("nº de features"); ax[1].legend(frameon=False); ax[1].set_title("Distribución de la pureza"); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Las *features* del SAE son más **monosemánticas** que las componentes de **PCA** (pureza media más alta) pero **no necesariamente que los centroides de k-means**, que optimizan directamente agrupar documentos. La ventaja del SAE no es ganar a k-means en pureza, sino que cada documento es una **suma de pocos átomos** (composicionalidad), lo que permite descomponer un vector en varios conceptos y editar uno de ellos. Mira también la cola: hay *features* mezcladas (pureza baja).

In [ ]:
cols = np.where(activos)[0]; pur = P["SAE TopK"]; orden = cols[np.argsort(-pur)][:8]
inv_voc = {j: t for t, j in lsi.tf.vocabulary_.items()} if lsi is not None else None
filas = []
for j in orden:
    top_docs = np.argsort(-Z[:, j])[:20]; faceta_dom = np.bincount(fac_id[top_docs]).argmax(); nombre_f = np.unique(facet)[faceta_dom]
    palabras = ""
    if lsi is not None:
        w = sae.W_dec.data[:, j].numpy() @ lsi.svd.components_; palabras = ", ".join(inv_voc[t] for t in np.argsort(-w)[:6])
    filas.append({"feature": int(j), "pureza": pur[list(cols).index(j)], "faceta dominante": nombre_f, "idioma dominante": pd.Series(lang[top_docs]).mode()[0], "términos del átomo (decoder × SVD)": palabras, "activa en %docs": 100 * float((Z[:, j] > 0).mean())})
display(pd.DataFrame(filas).set_index("feature").round(2))

> 👀 **Qué debes observar.** Las mejores *features* se leen como conceptos de la documentación («expiración, ttl, vencimiento…» ↔ faceta `cache.ttl`) y, según el átomo, agrupan términos de **uno o de los dos idiomas** (compara la columna de términos con la de idioma dominante: que el espacio comparta direcciones entre ES y EN es una propiedad del embedder, no del SAE). Esto es un **auto-etiquetado manual**: en la práctica se automatiza con un LLM («auto-interpretabilidad») que *también* hay que validar.

In [ ]:
# ¿Son estables las features entre semillas? (Paulo & Belrose 2025 lo midieron en LLMs; aquí en pequeño)
sae_b, _ = entrena_topk(E, n_lat=96, k=4, pasos=pasos, auxk=True, seed=1); Za, _ = codifica(sae_b, E)
Wa, Wb = sae.W_dec.data.T[activos], sae_b.W_dec.data.T[(Za.numpy() > 0).sum(0) >= 10]; cos = (Wa @ Wb.T).abs().max(1).values.numpy()
fig, ax = plt.subplots(figsize=(6, 3.2)); ax.hist(cos, bins=np.linspace(0, 1, 21), color=AZUL); ax.axvline(.9, color=ROJO, ls="--")
ax.set_xlabel("coseno máximo del átomo con los átomos de otra semilla"); ax.set_ylabel("nº de features"); ax.set_title(f"Estabilidad entre semillas: {np.mean(cos > .9):.0%} con coseno > 0.9"); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** **No todas** las features se reproducen entre semillas. Paulo & Belrose (2025) midieron en un SAE de 131 K latentes sobre Llama 3 8B que solo ≈ 30 % de las features se comparten entre semillas, y concluyen que el conjunto aprendido es *una* descomposición útil, **no la lista completa de lo que usa el modelo**. En nuestro juguete se reproduce **menos de la mitad** (≈ 0.4–0.5 según pasos y semilla; el título de la figura da el valor de tu ejecución) aunque los datos tengan solo 36 clústeres; la lección es la misma: **reporta varias semillas** antes de afirmar «el modelo tiene la feature X».

### 7.1 Steering con una *feature* del SAE (y comparación con la diferencia de medias)

In [ ]:
j_git = max(cols, key=lambda j: np.mean(topic[np.argsort(-Z[:, j])[:20]] == "git"))              # la feature más "git"
w_sae = sae.W_dec.data[:, j_git].numpy(); w_dm = (mu["git"] - mu["cache"]); w_dm = w_dm / np.linalg.norm(w_dm)
En_ = E / np.linalg.norm(E, axis=1, keepdims=True); res = {"átomo del SAE": [], "diferencia de medias": []}; alphas2 = [0, .25, .5, .75, 1, 1.5, 2]
for a in alphas2:
    for nombre, w in (("átomo del SAE", w_sae), ("diferencia de medias", w_dm)):
        q = En_[qs] + a * w; q /= np.linalg.norm(q, axis=1, keepdims=True); sc = q @ En_.T; sc[np.arange(len(qs)), qs] = -9
        res[nombre].append(float((topic[np.argsort(-sc, axis=1)[:, :10]] == "git").mean()))
fig, ax = plt.subplots(figsize=(6.8, 3.4)); ax.plot(alphas2, res["átomo del SAE"], "o-", color=VERDE, label=f"átomo SAE (feature {j_git})"); ax.plot(alphas2, res["diferencia de medias"], "s--", color=AZUL, label="diferencia de medias (git − cache)")
ax.set_xlabel("α (misma escala: vectores unitarios)"); ax.set_ylabel("% del top-10 que es «git»"); ax.legend(frameon=False); ax.set_title("Steering: ¿una feature aprendida gana a una media?"); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** La *feature* del SAE sí empuja los resultados hacia «git», pero **la diferencia de medias (sin entrenar nada) es igual o mejor**: coherente con *AxBench* (Wu et al., 2025), donde los SAE **no superan** a líneas base simples para steering/detección. Un SAE aporta **descomposición y legibilidad**, no necesariamente un mejor mando de control. 🔴 No lo vendas como lo segundo sin medirlo.

## 8. ¿Qué codifica un embedding de texto? Longitud y orden
Dos propiedades que a menudo se asumen: **longitud** y **orden de las palabras**. En LSI (bolsa de palabras) sabemos la respuesta *por construcción*; con un embedder real se mide igual.

In [ ]:
es_ = lang == "es"; lon = np.array([len(t.split()) for t in txt]); y_len = np.digitize(lon[es_], np.quantile(lon[es_], [1/3, 2/3]))
acc_len = sonda(E[es_], y_len)
# ¿POR QUÉ se lee la longitud? Confusor: la longitud depende de qué plantillas de frase salieron, y eso es léxico (palabras de función)
rng = np.random.default_rng(0); idx = rng.choice(len(txt), 200, replace=False); barajado = [" ".join(rng.permutation(txt[i].split())) for i in idx]
if lsi is not None:
    Zb = lsi.svd.transform(lsi.tf.transform(barajado)); Zb /= np.linalg.norm(Zb, axis=1, keepdims=True); cos_orden = float(np.mean(np.sum(Zb * E[idx], axis=1)))
else:
    cos_orden = float("nan")
print(f"sonda de LONGITUD en docs ES (3 terciles, azar 0.33): {acc_len:.2f}")
print(f"coseno entre un documento y el MISMO documento con las palabras barajadas: {cos_orden:.3f}  (1.0 = ciego al orden)")

> 👀 **Qué debes observar.** (1) La sonda **sí lee la longitud** (≈ 0.99, azar 0.33). ¿Significa que el embedding «codifica la longitud»? **No necesariamente**: en nuestro corpus la longitud depende de qué *plantillas de frase* se usaron (diferentes palabras de función), así que la sonda puede estar leyendo **léxico confundido con longitud**. Es el problema central del probing: **correlación ≠ lo que el modelo codifica por la razón que crees**; hace falta un control (p. ej. igualar el léxico y variar solo la longitud). (2) El **orden es invisible por construcción** (coseno 1.0): un embedder de bolsa de palabras no distingue «A antes de B». Los embedders de transformer sí codifican algo de orden y longitud (varía por modelo): **mídelo** con sondas controladas y con tests de permutación.

## 9. vec2text y qué revela un embedding (puente a privacidad)
**Morris, Kuleshov, Shmatikov y Rush (EMNLP 2023, *Text Embeddings Reveal (Almost) As Much As Text*)** muestran que se puede **reconstruir el texto** casi exactamente a partir de su embedding denso: su método multi-paso (generar, re-embeber, **corregir** iterativamente) recupera **el 92 % de textos de 32 tokens exactamente** y llega a reconstruir **nombres completos** en un conjunto de notas clínicas. **Zhang, Morris y Shmatikov (2025)** proponen inversión *zero-shot* universal entre embedders. Implicación: **un vector no es un hash**: guardar embeddings de datos sensibles equivale, en el peor caso, a guardar el texto.
Nuestro LSI es lineal: la inversión es una **multiplicación** ($x\approx V^\top z$). Medimos qué fracción de los 10 términos más altos recuperados **aparece realmente** en el documento y cómo cambia con **ruido** añadido a los embeddings (defensa) frente a la utilidad de retrieval.

In [ ]:
Vk = lsi.svd.components_ if lsi is not None else None
inv_voc = {j: t for t, j in lsi.tf.vocabulary_.items()} if lsi is not None else None
def prec_inversion(Zmat, idx, n_top=10):
    out = []
    for i in idx:
        est = Zmat[i] @ Vk; top = np.argsort(-est)[:n_top]; palabras = set(re.findall(r"\w+", txt[i].lower())); out.append(np.mean([inv_voc[j] in palabras for j in top]))
    return float(np.mean(out))
idx_i = np.random.default_rng(0).choice(len(txt), 200, replace=False); sigmas = [0, .02, .05, .1, .2, .4]; inv_p, util = [], []
rng_n = np.random.default_rng(1)
for s in sigmas:
    Zn = E + s * rng_n.standard_normal(E.shape).astype(np.float32); Znn = Zn / np.linalg.norm(Zn, axis=1, keepdims=True)
    inv_p.append(prec_inversion(Zn, idx_i)); qi = np.random.default_rng(2).choice(len(E), 300, replace=False); sc = E[qi] @ Znn.T; sc[np.arange(300), qi] = -9
    util.append(float((facet[np.argsort(-sc, axis=1)[:, :10]] == facet[qi][:, None]).mean()))
fig, ax = plt.subplots(figsize=(7, 3.6)); ax.plot(sigmas, inv_p, "o-", color=ROJO, label="inversión: precisión@10 de términos recuperados"); ax.plot(sigmas, util, "s-", color=VERDE, label="utilidad: precisión@10 de faceta (retrieval)")
ax.set_xlabel("desviación del ruido gaussiano por dimensión (embeddings de norma ≈ 1)"); ax.set_ylabel("valor"); ax.legend(frameon=False, fontsize=8); ax.set_ylim(0, 1.02); ax.set_title("Ruido como defensa: ¿cuánto cuesta?"); plt.tight_layout(); plt.show()
print(dict(zip(sigmas, zip(np.round(inv_p, 2), np.round(util, 2)))))

> 👀 **Qué debes observar.** Sin ruido, **≈ 90 % de los 10 términos recuperados están realmente en el documento** (el azar sería ≈ 1 %): el embedding filtra el vocabulario. Añadir ruido reduce la inversión, pero **también reduce la utilidad** de retrieval: es un **trade-off privacidad–utilidad** que debes medir con tu embedder y tu métrica (módulo 15). En un embedder real la inversión no es una multiplicación sino un modelo entrenado (vec2text) y las defensas se evalúan contra *ese* atacante, no contra el nuestro.
> 📌 Puente al **módulo 18**: política de qué datos se embeben, cifrado en reposo del índice, control de acceso, y no tratar los vectores como «anonimizados».

In [ ]:
RUN_VEC2TEXT = False    # True en Colab (GPU recomendada): pip install -q vec2text. NO probada en el entorno de desarrollo.
if RUN_VEC2TEXT:
    import vec2text
    corrector = vec2text.load_pretrained_corrector("gtr-base")                      # corrector para embeddings GTR-base (README del proyecto vec2text)
    recon = vec2text.invert_strings(["La caché expira tras un TTL de 60 segundos."], corrector=corrector, num_steps=20)
    print(recon)                                                                    # verifica nombres/argumentos en el README de la versión instalada
else:
    print("RUN_VEC2TEXT=False: celda omitida (opcional).")

## 🗺️ Mapa de madurez (estado a oct-2026)
Ya has hecho probing, steering, un SAE TopK y una auditoría de inversión; la tabla separa lo que sirve como diagnóstico fiable de lo que todavía es investigación antes de ver su uso en producción.


| Técnica | Madurez | Quién / dónde | Fuente |
|---|---|---|---|
| Probing con control tasks / selectividad | 🟢 sólido (como diagnóstico); 🟡 como *prueba causal* | NLP; probing crítico | Hewitt & Liang 2019; Belinkov 2022 |
| Structural probe (sintaxis en la geometría) | 🟢 sólido como resultado de 2019 en BERT/ELMo | Investigación | Hewitt & Manning 2019 |
| Hipótesis de representación lineal / direcciones de concepto | 🟡 emergente (evidencia fuerte, formalización reciente) | Representation engineering, steering | Park et al. 2023; Zou et al. 2023 |
| Steering por activaciones (suma de vectores) | 🟡 emergente | Investigación | Turner et al. 2023; Arditi et al. 2024 |
| Dimensiones dominantes / outliers en encoders | 🟢 sólido | BERT, GPT-2 | Timkey & van Schijndel 2021; Kovaleva et al. 2021 |
| **Sparse autoencoders** en LLMs (monosemanticidad, TopK, JumpReLU) | 🟡 emergente | Anthropic, OpenAI, Google DeepMind (Gemma Scope) | Bricken 2023; Gao et al. 2024; Rajamanoharan 2024; Lieberum 2024 |
| SAE sobre **embeddings de retrieval** | 🔴 experimental | Trabajos recientes (workshop/EMNLP) | Kang et al. 2024; Park et al. 2025 |
| SAE como método de steering/detección frente a baselines simples | 🔴 resultados negativos reportados | AxBench | Wu et al. 2025 |
| Inversión de embeddings (vec2text) | 🟡 emergente, amenaza **demostrada** en 32 tokens | Seguridad/privacidad | Morris et al. 2023; Zhang et al. 2025 |

## 10. 🏭 En producción
- **Diagnóstico geométrico barato en cada migración de embedder:** varianza por dimensión, coseno medio entre textos al azar, sondas de idioma/dominio. Un cambio de modelo que introduce una dimensión dominante o anisotropía extrema es una causa frecuente de regresiones silenciosas.
- **Filtros y routing con sondas:** una sonda lineal sobre el embedding (idioma, tema, PII) es una forma barata de enrutar o filtrar sin llamar a otro modelo; la selectividad y la deriva de la sonda se monitorizan como cualquier modelo.
- **Interpretabilidad de resultados:** con un SAE (o con k-means) puedes explicar *por qué* dos documentos se parecen («comparten las features expiración + ttl»); es una herramienta de **depuración y auditoría**, no una garantía causal.
- **Privacidad:** trata los embeddings como datos sensibles (inversión demostrada en 32 tokens). Control de acceso al índice, minimización de PII antes de embeber, y pruebas de inversión sobre tu propio embedder antes de afirmar «anonimizado».
- **Steering:** en producción rara vez se «gobierna» el retrieval con una feature: es más seguro usar filtros por metadatos o reranking. Considéralo experimental.

## 11. 🧠 Secretos de la élite
1. **Siempre reporta la tarea de control y la selectividad** (Hewitt & Liang). Una sonda MLP «lee» hasta etiquetas aleatorias por tipo (nuestro experimento: ≈ 0.95 en el control).
2. **Probar que se puede leer no prueba que se use.** Borrar linealmente el idioma dejó la sonda en el azar y el retrieval **intacto** (§4). Lo causal exige intervenir y medir comportamiento.
3. **Compara siempre con la diferencia de medias.** En *AxBench* los SAE no superaron a baselines simples en steering y detección; en nuestro juguete la media también igualó al átomo del SAE.
4. **Más latentes ≠ más verdad.** Las features dependen de tamaño del diccionario, $k$ y **semilla** (Paulo & Belrose: ≈ 30 % compartidas en un SAE de 131 K latentes sobre Llama 3 8B). Entrena ≥ 3 semillas y reporta la fracción estable.
5. **TopK te quita un hiperparámetro difícil** ($\lambda$ de L1) y fija $L_0$; mide siempre la frontera NMSE–$L_0$ y los latentes muertos (AuxK, inicialización *tied*, decoder normalizado).
6. **Una dimensión dominante puede arruinar el coseno.** Diagnostica `var_top1` y el coseno medio de pares al azar al elegir/migrar embedder (Timkey & van Schijndel; Kovaleva et al.: esas dimensiones son críticas para el modelo, no «basura» que se pueda borrar). Su primo en retrieval es la **hubness** (§5b): documentos que salen en todas las listas y anti-hubs que nadie encuentra; mide la cola de $N_k$ y prueba centrado o CSLS antes de culpar al modelo.
7. **Un embedding no es un hash.** vec2text reconstruye el 92 % de textos de 32 tokens (Morris et al.); evalúa ruido/cuantización como defensa **contra un atacante real**, no con una heurística.
8. **Valida los auto-etiquetados.** Etiquetar features con un LLM es útil pero debe contrastarse con pureza medida sobre etiquetas conocidas, como hicimos con las facetas.

## 12. ⚠️ Errores comunes
- Concluir que «el embedding codifica sintaxis/tema» con una sonda sobreparametrizada y sin control task.
- Interpretar los **ejes** del embedding como conceptos (están rotados y en superposición).
- Medir interpretabilidad de un SAE con *ejemplos elegidos a mano* en lugar de una métrica con verdad terreno o muestreo aleatorio de features.
- Olvidar normalizar las columnas del decoder (el SAE «hace trampa» escalando).
- Comparar SAEs con distinto $L_0$ sin igualar la frontera.
- Tratar la dirección de un concepto como **causal** sin intervención ni baseline.
- Dar por «anónimos» los embeddings de datos con PII.

## 13. ✅ Autoevaluación
<details><summary><b>1. (Cálculo)</b> Una sonda lineal saca 0.97 en la tarea real y 0.55 en la de control; una MLP 0.995 y 0.95. Calcula la selectividad de cada una e interpreta.</summary>

Lineal: 0.97 − 0.55 = **0.42**. MLP: 0.995 − 0.95 = **0.045**. La MLP memoriza (tiene capacidad para aprender la tarea sin ayuda del embedding): su acierto no prueba que el embedding codifique el concepto; la lineal es más fiable.
</details>

<details><summary><b>2. (Cálculo)</b> Un SAE TopK con 256 latentes y k=4 sobre vectores de 64 dimensiones: ¿cuántos parámetros tiene (sin sesgos) y cuántos valores no nulos por muestra?</summary>

$W_{enc}$: $256\times64=16\,384$; $W_{dec}$: $64\times256=16\,384$ ⇒ 32 768 parámetros (más sesgos: 256 + 64). **4** no nulos por muestra: $L_0=k$ exacto.
</details>

<details><summary><b>3. (Diagnóstico)</b> Tras migrar de embedder, el coseno medio entre textos aleatorios pasa de 0.15 a 0.85 y nDCG@10 cae 6 puntos. Hipótesis y comprobación.</summary>

Anisotropía por una o pocas **dimensiones dominantes** (Timkey & van Schijndel). Comprueba `var_top1`, el coseno medio estandarizado y el nDCG tras **estandarizar/whitening**. Si se recupera, la causa era la geometría y no el contenido semántico.
</details>

<details><summary><b>4. (Diagnóstico)</b> Tu SAE tiene una feature «refund policy» muy interpretable, pero al sumar su átomo a la consulta el retrieval casi no cambia. ¿Por qué podría pasar?</summary>

Interpretable ≠ causal. El átomo puede describir la actividad pero no ser la dirección que usa la similitud; la escala del átomo puede ser pequeña respecto a la norma de la consulta; y una dirección de **diferencia de medias** podría funcionar mejor (AxBench). Mide la intervención y compara con baselines.
</details>

<details><summary><b>5. (Transferencia)</b> Un cliente legal quiere embeber contratos con datos personales. ¿Qué le recomiendas a la luz de vec2text?</summary>

Tratar los vectores como datos sensibles: minimizar/redactar PII antes de embeber, cifrar el índice y limitar acceso, evaluar la inversión sobre *su* embedder (vec2text y variantes), considerar embeber solo fragmentos sin identificadores, y medir el trade-off ruido/utilidad (módulo 18).
</details>

<details><summary><b>6. (Transferencia)</b> ¿Cómo comprobarías si una feature de un SAE entrenado sobre un embedder real es «la misma» que otra de un SAE con otra semilla?</summary>

Compara los átomos del decoder por coseno máximo (y sus activaciones sobre un conjunto común); marca como estables las de coseno > 0.9 y compara su *top-activating* examples. Reporta la fracción estable y repite con ≥ 3 semillas.
</details>

## 14. 📚 Referencias
- Hewitt & Manning (2019). *A Structural Probe for Finding Syntax in Word Representations*. NAACL 2019. <https://aclanthology.org/N19-1419/>
- Hewitt & Liang (2019). *Designing and Interpreting Probes with Control Tasks*. EMNLP-IJCNLP 2019. <https://arxiv.org/abs/1909.03368>
- Belinkov (2022). *Probing Classifiers: Promises, Shortcomings, and Advances*. Computational Linguistics 48(1). <https://arxiv.org/abs/2102.12452>
- Park, Choe, Veitch (2023/2024). *The Linear Representation Hypothesis and the Geometry of Large Language Models*. ICML 2024. <https://arxiv.org/abs/2311.03658>
- Turner et al. (2023). *Steering Language Models With Activation Engineering*. Preprint. <https://arxiv.org/abs/2308.10248> · Zou et al. (2023). *Representation Engineering: A Top-Down Approach to AI Transparency*. Preprint. <https://arxiv.org/abs/2310.01405> · Arditi et al. (2024). *Refusal in Language Models Is Mediated by a Single Direction*. NeurIPS 2024. <https://arxiv.org/abs/2406.11717>
- Belrose et al. (2023). *LEACE: Perfect linear concept erasure in closed form*. NeurIPS 2023. <https://arxiv.org/abs/2306.03819>
- Timkey & van Schijndel (2021). *All Bark and No Bite: Rogue Dimensions in Transformer Language Models Obscure Representational Quality*. EMNLP 2021. <https://arxiv.org/abs/2109.04404> · Kovaleva et al. (2021). *BERT Busters: Outlier Dimensions that Disrupt Transformers*. Findings of ACL 2021. <https://arxiv.org/abs/2105.06990> · Ethayarajh (2019). *How Contextual are Contextualized Word Representations?* EMNLP-IJCNLP 2019. <https://arxiv.org/abs/1909.00512> · Mu, Bhat, Viswanath (2018). *All-but-the-Top: Simple and Effective Postprocessing for Word Representations*. ICLR 2018. <https://arxiv.org/abs/1702.01417>
- Radovanović, Nanopoulos, Ivanović (2010). *Hubs in Space: Popular Nearest Neighbors in High-Dimensional Data*. JMLR 11. <https://jmlr.org/papers/v11/radovanovic10a.html>
- Conneau, Lample, Ranzato, Denoyer, Jégou (2018). *Word Translation Without Parallel Data* (CSLS). ICLR 2018. <https://arxiv.org/abs/1710.04087>
- Elhage et al. (2022). *Toy Models of Superposition*. Transformer Circuits Thread (Anthropic). <https://arxiv.org/abs/2209.10652>
- Bricken et al. (2023). *Towards Monosemanticity: Decomposing Language Models With Dictionary Learning*. Anthropic. <https://transformer-circuits.pub/2023/monosemantic-features> · Templeton et al. (2024). *Scaling Monosemanticity*. Anthropic. <https://transformer-circuits.pub/2024/scaling-monosemanticity/>
- Cunningham et al. (2023). *Sparse Autoencoders Find Highly Interpretable Features in Language Models*. ICLR 2024. <https://arxiv.org/abs/2309.08600>
- Makhzani & Frey (2013). *k-Sparse Autoencoders*. ICLR 2014. <https://arxiv.org/abs/1312.5663> · Gao et al. (2024). *Scaling and evaluating sparse autoencoders* (OpenAI). ICLR 2025. <https://arxiv.org/abs/2406.04093> · Rajamanoharan et al. (2024). *Jumping Ahead: Improving Reconstruction Fidelity with JumpReLU Sparse Autoencoders*. Preprint. <https://arxiv.org/abs/2407.14435> · Lieberum et al. (2024). *Gemma Scope: Open Sparse Autoencoders Everywhere All At Once on Gemma 2*. Preprint. <https://arxiv.org/abs/2408.05147>
- Paulo & Belrose (2025). *Sparse Autoencoders Trained on the Same Data Learn Different Features*. Preprint. <https://arxiv.org/abs/2501.16615> · Wu et al. (2025). *AxBench: Steering LLMs? Even Simple Baselines Outperform Sparse Autoencoders*. ICML 2025. <https://arxiv.org/abs/2501.17148> · Karvonen et al. (2025). *SAEBench: A Comprehensive Benchmark for Sparse Autoencoders in Language Model Interpretability*. ICML 2025. <https://arxiv.org/abs/2503.09532>
- SAE sobre embeddings de retrieval: Kang, Wang, Xiong (2024). *Interpret and Control Dense Retrieval with Sparse Latent Features*. Preprint. <https://arxiv.org/abs/2411.00786> · Park, Kim, Ko (2025). *Decoding Dense Embeddings: Sparse Autoencoders for Interpreting and Discretizing Dense Retrieval*. EMNLP 2025. <https://arxiv.org/abs/2506.00041>
- Morris, Kuleshov, Shmatikov, Rush (2023). *Text Embeddings Reveal (Almost) As Much As Text*. EMNLP 2023. <https://arxiv.org/abs/2310.06816> · Li, Xu, Song (2023). *Sentence Embedding Leaks More Information than You Expect: Generative Embedding Inversion Attack to Recover the Whole Sentence*. Findings of ACL 2023. <https://arxiv.org/abs/2305.03010> · Zhang, Morris, Shmatikov (2025). *Universal Zero-shot Embedding Inversion*. Preprint. <https://arxiv.org/abs/2504.00147>
- Material del curso: módulos 09–12 (geometría y embedders), 15 (`poliglota_eval.py`), 18 (privacidad y producción).